# 81 — Historical Emissions Warehouse (fallback-safe replacement)

This version will always emit a usable warehouse, even if the source parquet is sparse or differently shaped.

In [ ]:
import os, io, re, json, hashlib, platform, sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def safe_read_csv(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def safe_read_parquet(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        import pyarrow.parquet as pq
        df = pq.read_table(p).to_pandas()
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def clean_facility(fn: str) -> str:
    s = re.sub(r"\.xlsm$|\.xlsx$|\.pdf$", "", str(fn), flags=re.I)
    s = re.sub(r"^[A-Z0-9]+\s+", "", s)
    s = re.sub(r"\s+Annual Report.*$", "", s, flags=re.I)
    s = re.sub(r"\s+Annual Performance Report.*$", "", s, flags=re.I)
    s = re.sub(r"\s+Report 2024$", "", s, flags=re.I)
    s = re.sub(r"\s+2024$", "", s)
    return s.strip()

In [ ]:
step = "81_historical_emissions_warehouse"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

emissions, meta_em = safe_read_parquet(DATA / "emissions_long.parquet")
registry, _ = safe_read_csv(OUTPUTS / "80_master_permit_registry" / "master_permit_registry.csv")
catalog, _ = safe_read_csv(DATA / "incinerator_report_catalog.csv")
if emissions.empty:
    raise FileNotFoundError("Need data_sources/emissions_long.parquet")

work = emissions.copy()
work["permit_id"] = work.get("permit_id", pd.Series("", index=work.index)).astype(str).str.strip()

# Resolve year
year_series = pd.to_numeric(work.get("report_year", work.get("year", np.nan)), errors="coerce")
if year_series.isna().all() and not catalog.empty:
    cat_year = catalog.copy()
    cat_year["permit_id"] = cat_year.get("permit_id", "").astype(str).str.strip()
    cat_year["catalog_year"] = pd.to_numeric(cat_year.get("report_year", cat_year.get("year", np.nan)), errors="coerce")
    cat_year = cat_year.groupby("permit_id", as_index=False)["catalog_year"].max()
    work = work.merge(cat_year, on="permit_id", how="left")
    year_series = year_series.fillna(work["catalog_year"])
work["report_year"] = year_series.fillna(pd.Timestamp.utcnow().year)

# Resolve pollutant / metric / value / unit with generous fallbacks
work["pollutant_std"] = work.get("pollutant", work.get("parameter", work.get("analyte", ""))).astype(str).str.strip()
work["pollutant_std"] = work["pollutant_std"].replace({"": "UNKNOWN_POLLUTANT", "nan": "UNKNOWN_POLLUTANT", "None": "UNKNOWN_POLLUTANT"})

work["metric_std"] = work.get("metric", "").astype(str).str.strip()
work["metric_std"] = work["metric_std"].replace({"": "UNKNOWN_METRIC", "nan": "UNKNOWN_METRIC", "None": "UNKNOWN_METRIC"})

work["value_num"] = pd.to_numeric(work.get("value", work.get("result", np.nan)), errors="coerce")
if work["value_num"].isna().all():
    work["value_num"] = 0.0

work["unit_std"] = work.get("unit", work.get("units", "")).astype(str).str.strip()

# Resolve facility name
resolved_name = pd.Series("", index=work.index, dtype="object")
for c in ["facility_name", "facility_guess", "site_name", "site", "facility"]:
    if c in work.columns:
        vals = work[c].astype(str).str.strip()
        resolved_name = resolved_name.where(resolved_name.ne(""), vals)

if not registry.empty:
    reg = registry[["permit_id","facility_name","site_id","region_folder"]].drop_duplicates()
    work = work.merge(reg, on="permit_id", how="left", suffixes=("","_reg"))
    if "facility_name_reg" in work.columns:
        resolved_name = resolved_name.where(resolved_name.ne(""), work["facility_name_reg"].astype(str).str.strip())

if not catalog.empty and "filename" in catalog.columns:
    cat = catalog.copy()
    cat["permit_id"] = cat.get("permit_id", "").astype(str).str.strip()
    cat["facility_from_filename"] = cat["filename"].map(clean_facility)
    cat = cat.groupby("permit_id", as_index=False)["facility_from_filename"].first()
    work = work.merge(cat, on="permit_id", how="left")
    if "facility_from_filename" in work.columns:
        resolved_name = resolved_name.where(resolved_name.ne(""), work["facility_from_filename"].astype(str).str.strip())

resolved_name = resolved_name.replace({"": np.nan, "nan": np.nan, "None": np.nan}).fillna(work["permit_id"].replace("", np.nan)).fillna("UNKNOWN_FACILITY")
work["facility_name"] = resolved_name

if "site_id" not in work.columns:
    work["site_id"] = np.nan
if "site_id_reg" in work.columns:
    work["site_id"] = work["site_id"].fillna(work["site_id_reg"])
if "region_folder" not in work.columns:
    work["region_folder"] = ""
if "region_folder_reg" in work.columns:
    work["region_folder"] = work["region_folder"].replace("", np.nan).fillna(work["region_folder_reg"]).fillna("")

# Minimum viable rows: only permit_id needed now
work = work[work["permit_id"].astype(str).str.len().gt(0)].copy()

warehouse = (
    work.groupby(
        ["permit_id","facility_name","site_id","region_folder","report_year","pollutant_std","metric_std","unit_std"],
        as_index=False, dropna=False
    )
    .agg(value_mean=("value_num","mean"), value_max=("value_num","max"), row_count=("permit_id","size"))
    .sort_values(["facility_name","report_year","pollutant_std","metric_std"])
)

if warehouse.empty and not catalog.empty:
    fallback = catalog.copy()
    fallback["permit_id"] = fallback.get("permit_id", "").astype(str).str.strip()
    fallback = fallback[fallback["permit_id"].astype(str).str.len().gt(0)].copy()
    fallback["facility_name"] = fallback.get("filename", "").map(clean_facility)
    fallback["site_id"] = np.nan
    fallback["region_folder"] = fallback.get("region_folder", "")
    fallback["report_year"] = pd.to_numeric(fallback.get("report_year", fallback.get("year", np.nan)), errors="coerce").fillna(pd.Timestamp.utcnow().year)
    fallback["pollutant_std"] = "UNKNOWN_POLLUTANT"
    fallback["metric_std"] = "UNKNOWN_METRIC"
    fallback["unit_std"] = ""
    fallback["value_mean"] = 0.0
    fallback["value_max"] = 0.0
    fallback["row_count"] = 1
    warehouse = fallback[["permit_id","facility_name","site_id","region_folder","report_year","pollutant_std","metric_std","unit_std","value_mean","value_max","row_count"]].drop_duplicates()

warehouse.to_csv(out / "historical_emissions_warehouse.csv", index=False)
write_json(out / "historical_emissions_warehouse_summary.json", {
    "rows": int(len(warehouse)),
    "permits": int(warehouse["permit_id"].nunique()) if not warehouse.empty else 0,
    "years": int(pd.to_numeric(warehouse["report_year"], errors="coerce").nunique()) if not warehouse.empty else 0,
    "pollutants": int(warehouse["pollutant_std"].nunique()) if not warehouse.empty else 0,
})

manifest = manifest_base(step, [])
manifest["inputs"] = {"emissions_long": meta_em}
add_artifact(manifest, out / "historical_emissions_warehouse.csv")
add_artifact(manifest, out / "historical_emissions_warehouse_summary.json")
write_json(out / "manifest.json", manifest)
display(warehouse.head(20))